# 🪟 Day 07: Sliding Window — The O(n) Substring Master

---

## Two Flavors, One Idea

Sliding window is about maintaining a **window** (a contiguous subarray or substring) that slides across the input. Instead of recalculating from scratch each time, you adjust the window by adding/removing elements at the edges.

**Two types:**

1. **Fixed window:** Size is given (e.g., "max sum of subarray of size k")
2. **Variable window:** Size changes dynamically (e.g., "shortest substring containing all characters")

The variable window is the one that shows up in interviews constantly. Here's the template:

```
left = 0
for right in range(n):
    # EXPAND: add nums[right] to window
    
    while window_is_invalid:
        # SHRINK: remove nums[left] from window
        left += 1
    
    # UPDATE answer (window is [left..right])
```

---

## 🎯 Problem 1: Maximum Sum Subarray of Size K (Warm-up)

Fixed window — the simplest sliding window.

```
Input: [2, 1, 5, 1, 3, 2], k = 3  →  Output: 9 (subarray [5,1,3])
```

In [ ]:
def maxSumFixedWindow(nums, k):
    """
    Fixed sliding window of size k.
    Time: O(n), Space: O(1)
    """
    window_sum = sum(nums[:k])
    max_sum = window_sum
    
    for i in range(k, len(nums)):
        window_sum += nums[i] - nums[i - k]  # Slide: add right, remove left
        max_sum = max(max_sum, window_sum)
    
    return max_sum

print(maxSumFixedWindow([2, 1, 5, 1, 3, 2], 3))  # 9

---

## 🎯 Problem 2: Longest Substring Without Repeating Characters (LC #3)

One of the most popular interview questions period.

```
Input: "abcabcbb"  →  Output: 3 ("abc")
Input: "bbbbb"     →  Output: 1
Input: "pwwkew"    →  Output: 3 ("wke")
```

**Variable window:** Expand right. When we see a duplicate, shrink from left until no duplicates.

In [ ]:
def lengthOfLongestSubstring(s):
    """
    Variable window + set for tracking characters.
    Time: O(n), Space: O(min(n, 26))
    """
    char_set = set()
    left = 0
    max_len = 0
    
    for right in range(len(s)):
        # Shrink window until no duplicate
        while s[right] in char_set:
            char_set.remove(s[left])
            left += 1
        
        char_set.add(s[right])
        max_len = max(max_len, right - left + 1)
    
    return max_len

print(lengthOfLongestSubstring("abcabcbb"))  # 3
print(lengthOfLongestSubstring("bbbbb"))      # 1
print(lengthOfLongestSubstring("pwwkew"))     # 3

# Optimized version: jump left directly using a dict
def lengthOfLongestSubstring_v2(s):
    last_seen = {}  # char → last index
    left = 0
    max_len = 0
    
    for right, ch in enumerate(s):
        if ch in last_seen and last_seen[ch] >= left:
            left = last_seen[ch] + 1  # Jump left past the duplicate
        last_seen[ch] = right
        max_len = max(max_len, right - left + 1)
    
    return max_len

print(lengthOfLongestSubstring_v2("abcabcbb"))  # 3

---

## 🎯 Problem 3: Minimum Window Substring (LC #76)

The **hardest** sliding window. If you can do this one, you can do any of them.

```
Find smallest substring of s that contains all characters of t.
Input: s = "ADOBECODEBANC", t = "ABC"  →  Output: "BANC"
```

**Strategy:** Expand right until you have all chars of t. Then shrink left to minimize. Track counts with a dict.

In [ ]:
from collections import Counter

def minWindow(s, t):
    """
    Minimum Window Substring
    Time: O(n), Space: O(k) where k = unique chars in t
    """
    if not t or not s:
        return ""
    
    need = Counter(t)       # What we need
    missing = len(t)        # How many chars still missing
    left = 0
    result = (0, float('inf'))  # (start, end) of best window
    
    for right, ch in enumerate(s):
        # EXPAND: if this char is needed, decrement missing
        if need[ch] > 0:
            missing -= 1
        need[ch] -= 1
        
        # SHRINK: while we have everything, try to minimize
        while missing == 0:
            # Update best window
            if right - left < result[1] - result[0]:
                result = (left, right)
            
            # Remove left char from window
            need[s[left]] += 1
            if need[s[left]] > 0:
                missing += 1
            left += 1
    
    start, end = result
    return s[start:end+1] if end != float('inf') else ""

print(minWindow("ADOBECODEBANC", "ABC"))  # "BANC"
print(minWindow("a", "a"))                 # "a"
print(minWindow("a", "aa"))                # ""

---

## 🎯 Problem 4: Longest Repeating Character Replacement (LC #424)

Sneaky sliding window problem that's super popular.

```
You can replace at most k characters. Find longest substring with all same chars.
Input: s = "AABABBA", k = 1  →  Output: 4 ("AABA" → replace B → "AAAA")
```

**Insight:** Window is valid if `window_size - max_freq_in_window <= k` (replacements needed ≤ k).

In [ ]:
def characterReplacement(s, k):
    """
    Sliding window: valid if (window_size - max_freq) <= k
    Time: O(n), Space: O(26) = O(1)
    """
    count = {}      # char → frequency in current window
    left = 0
    max_freq = 0    # Max frequency of any single char in window
    result = 0
    
    for right in range(len(s)):
        count[s[right]] = count.get(s[right], 0) + 1
        max_freq = max(max_freq, count[s[right]])
        
        # Window invalid: need more than k replacements
        window_size = right - left + 1
        if window_size - max_freq > k:
            count[s[left]] -= 1
            left += 1
        
        result = max(result, right - left + 1)
    
    return result

print(characterReplacement("ABAB", 2))     # 4
print(characterReplacement("AABABBA", 1))  # 4

---

## 🗺️ Sliding Window Pattern Map

```
╔══════════════════════════════════════════════════════════════════════╗
║  SLIDING WINDOW PATTERNS                                            ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  1. FIXED WINDOW: size given, slide by adding right, removing left  ║
║     → Max sum of size k, max average                                 ║
║                                                                      ║
║  2. VARIABLE — LONGEST: expand right, shrink left when invalid      ║
║     → Longest substring without repeating, with at most k distinct   ║
║                                                                      ║
║  3. VARIABLE — SHORTEST: expand until valid, shrink to minimize     ║
║     → Minimum window substring, minimum size subarray sum            ║
║                                                                      ║
║  Common tools: set (unique check), dict/Counter (frequency),        ║
║                 missing counter (how many chars still needed)         ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

## 📝 Practice Problems

| # | Problem | Difficulty | Type |
|---|---------|-----------|------|
| 3 | Longest Substring Without Repeating | Medium | Variable (longest) |
| 76 | Minimum Window Substring | Hard | Variable (shortest) |
| 424 | Longest Repeating Char Replacement | Medium | Variable (longest) |
| 567 | Permutation in String | Medium | Fixed window + freq |
| 438 | Find All Anagrams in a String | Medium | Fixed window + freq |
| 209 | Min Size Subarray Sum | Medium | Variable (shortest) |
| 239 | Sliding Window Maximum | Hard | Monotonic deque |

---

## ✅ Key Takeaways

```
┌──────────────────────────────────────────────────────────────────────┐
│  📌 Fixed window: add right, remove left, constant size             │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Variable window: expand right, shrink left while invalid        │
├──────────────────────────────────────────────────────────────────────┤
│  📌 "Longest" → shrink when invalid. "Shortest" → shrink when valid │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Use set for uniqueness, Counter/dict for frequency tracking     │
├──────────────────────────────────────────────────────────────────────┤
│  📌 min_window_substring is the gold standard — master it           │
└──────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: String Patterns

String-specific tricks, palindromes, and string building.

---

### 🎉 Sliding Window Complete!
Fixed or variable, expand or shrink — you've got the template.